## Подготовка окружения

In [82]:
import numpy as np
import torch
import re

from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.normalizers import Sequence, NFKC, Lowercase
from tokenizers.pre_tokenizers import Sequence as PreTokenSequence
from tokenizers.pre_tokenizers import Whitespace, Punctuation, Digits

from torch.nn.utils.rnn import pad_sequence

## Загрузка данных

In [63]:
with open("data/TinyStories-train.txt", "r", encoding="utf-8") as file:
    text = file.read()

train_corpus = text.split('<|endoftext|>')

In [64]:
train_corpus[:5]

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.\n',
 '\nOnce upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were

In [65]:
with open("data/TinyStories-valid.txt", "r") as file:
    text = file.read()

valid_corpus = text.split('<|endoftext|>')

In [66]:
valid_corpus[:5]

[' Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."\nAfter playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.\n',
 '\nOnce upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.\nRoxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy was sad. She wanted to climb the icy hill so much. Then, she saw a little bird named Billy. Billy saw that Roxy was sad and asked, "Why are you sad, Roxy?"\nRoxy told Billy about the icy hill and how she couldn\'t climb it. Billy said, "I have an idea! Let\'s f

## Предварительная обработка

Чтобы обучать decoder-only Transformer на наших данных, нужно очистить текст, преобразовать в токены, а затем преобразовать токены в эмбеддинги.

### Очистка сырого текста

Для классификации и генерации текст чистят по-разному, потому что у моделей разные цели.

В классификации модель должна понять общий смысл текста и отнести его к какому-то классу: например, отзыв положительный или отрицательный, письмо спам или не спам, сообщение токсичное или нормальное. Поэтому там можно сильнее упрощать текст: убирать лишние символы, иногда пунктуацию, приводить слова к нижнему регистру, удалять стоп-слова. Главное — сохранить смысл.

А в генерации модель должна не просто понять смысл, а научиться писать текст. То есть она должна уметь ставить запятые, точки, вопросительные знаки, сохранять порядок слов, структуру предложений и стиль. Поэтому для decoder-only модели нельзя слишком сильно чистить текст. Если убрать пунктуацию, модель потом и сама будет генерировать текст без нормальных точек и запятых.

Не будем учить модель абзацам, поэтому, заменим `\n`, `\t`, красивые кавычки на пробельный символ.

In [67]:
def clean_text(text: str) -> str:
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [68]:
train_corpus = [clean_text(doc) for doc in train_corpus]
valid_corpus = [clean_text(doc) for doc in valid_corpus]

In [69]:
train_corpus[:5]

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt. Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt." Together, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong. One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were fallin

## Токенизация

Токенизацию будем выполнять на уровне слов с помощью библиотеки `tokenizers` от Hugging Face. В процессе обучения токенизатора на обучающем корпусе сразу будет построен словарь токенов `token -> id`.

Обязательно сделаем специальные токены: `<bos> - Beginning Of Sequence`, `<eos> — End Of Sequence`, `<pad> — Padding`, `<unk> — Unknown`

Используем такой конвейер токенизации:

1) Normalization — `NFKC`, `Lowercase`

    NFKC — нормализует Unicode-символы.

    Lowercase — переводит текст в нижний регистр.

2) Pre-tokenization — `Whitespace`, `Punctuation`, `Digits`

    Whitespace — разбивает текст по пробелам и границам слов.

    Punctuation — отделяет знаки препинания от слов.

    Digits — отделяет числа от букв.

3) Model — `WordLevel`

    WordLevel — сопоставляет каждый токен с числовым индексом из словаря. Если токена нет в словаре, он заменяется на `<unk>`.

Возьмем размер словаря 20 000 самых частых токенов. Все остальные редкие токены, которые не попадут в словарь, будут заменяться специальным токеном `<unk>`.

In [70]:
tokenizer = Tokenizer(model=WordLevel(unk_token="<unk>"))
tokenizer.normalizer = Sequence([
    NFKC(), Lowercase()
    ])
tokenizer.pre_tokenizer = PreTokenSequence([
    Whitespace(), 
    Punctuation(), 
    Digits()
    ])
trainer = WordLevelTrainer(
    vocab_size=20000,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]
)

tokenizer.train_from_iterator(train_corpus, trainer=trainer)  # pyright: ignore[reportUnknownMemberType]

Проверим работу токенизатора.

In [71]:
print(tokenizer.get_vocab_size()) # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]
print(tokenizer.encode('One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp.').tokens) # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]
print(tokenizer.encode('One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp.').ids) # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]

20000
['one', 'day', ',', 'a', 'little', 'girl', 'named', 'lily', 'found', 'a', 'needle', 'in', 'her', 'room', '.', 'she', 'knew', 'it', 'was', 'difficult', 'to', 'play', 'with', 'it', 'because', 'it', 'was', 'sharp', '.']
[35, 28, 7, 9, 39, 53, 77, 24, 112, 9, 1776, 21, 16, 188, 4, 13, 160, 14, 11, 1315, 8, 49, 23, 14, 192, 14, 11, 864, 4]


Выведем весь наш словарь на 20000 токенов.

In [93]:
vocab = tokenizer.get_vocab() # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
vocab

{'justin': 6113,
 'positive': 5155,
 'yvonne': 13190,
 'bleating': 10951,
 'coffin': 18808,
 'meats': 11232,
 'gio': 12092,
 'alexis': 10515,
 'sprocket': 16746,
 'snuggly': 6790,
 'hesitant': 3863,
 'jemima': 7579,
 'biplane': 18766,
 'today': 547,
 'plums': 7959,
 'sailors': 3964,
 'textures': 5345,
 'spyglass': 12054,
 'remember': 623,
 'introduced': 3294,
 'robert': 3718,
 'plopped': 5516,
 'string': 1001,
 'matt': 3371,
 'boar': 8985,
 'yelling': 3130,
 'stocking': 11568,
 'cradling': 11940,
 'coos': 10067,
 'tube': 2077,
 'sairy': 17765,
 'science': 5726,
 'reunite': 9501,
 'yoyo': 13189,
 'chimes': 11722,
 'gathered': 1143,
 'postman': 4070,
 'pa': 8045,
 'know': 169,
 'discussions': 15872,
 'marvel': 6915,
 'technician': 10334,
 'ukulele': 17135,
 'ups': 2566,
 'clenched': 9688,
 'almost': 1056,
 'abba': 12482,
 'loveable': 13906,
 'fiery': 8101,
 'tegan': 18661,
 'shuffle': 11990,
 'plip': 17032,
 'pulls': 1981,
 'potential': 6892,
 'lives': 2260,
 'plowed': 13498,
 'baddy': 1

Каждый документ в обучающем и валидационном корпусе преобразуем в последовательность индексов токенов. Для этого сначала применяем обученный токенизатор, затем добавляем специальные токены <bos> и <eos>. После этого все последовательности приводим к одной длине с помощью padding.

In [79]:
bos_id = tokenizer.token_to_id('<bos>') # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
eos_id = tokenizer.token_to_id('<eos>') # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
print(bos_id, eos_id) # pyright: ignore[reportUnknownArgumentType]

2 3


In [75]:
train_tokens = [[bos_id] + tokenizer.encode(doc).ids + [eos_id] for doc in train_corpus] # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
valid_tokens = [[bos_id] + tokenizer.encode(doc).ids + [eos_id] for doc in valid_corpus] # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]

### Сделаем паддинг последовательностей с помощью метода pad_sequence

In [95]:
tensor_train = [torch.tensor(doc, dtype=torch.long) for doc in train_tokens] # pyright: ignore[reportUnknownVariableType]
padded_train = pad_sequence(sequences=tensor_train, batch_first=True, padding_value=vocab["<pad>"]) # pyright: ignore[reportUnknownArgumentType]

In [96]:
tensor_valid = [torch.tensor(doc, dtype=torch.long) for doc in valid_tokens] # pyright: ignore[reportUnknownVariableType]
padded_valid = pad_sequence(sequences=tensor_valid, batch_first=True, padding_value=vocab["<pad>"]) # pyright: ignore[reportUnknownArgumentType]

In [102]:
print(padded_train)
print(padded_train.shape)

tensor([[ 2, 35, 28,  ...,  0,  0,  0],
        [ 2, 47, 52,  ...,  0,  0,  0],
        [ 2, 35, 28,  ...,  0,  0,  0],
        ...,
        [ 2, 47, 52,  ...,  0,  0,  0],
        [ 2, 47, 52,  ...,  0,  0,  0],
        [ 2, 47, 52,  ...,  0,  0,  0]])
torch.Size([2119719, 1148])
